In [ ]:
from google.colab import drive
from sklearn import preprocessing

In [ ]:
drive.mount('/content/drive')

# Read the images and generate the train and test dataset

In [ ]:
import os
os.chdir('/content/drive/My Drive/ComputerVision_with_CNNs/Plant_Breed')

In [ ]:
#import pdb; pdb.set_trace()
from zipfile import ZipFile
with ZipFile('train.zip', 'r') as z:
  z.extractall()

In [ ]:
from zipfile import ZipFile
with ZipFile('test.zip', 'r') as z:
  z.extractall()

In [ ]:
x_test=[]
os.chdir('/content/drive/My Drive/ComputerVision_with_CNNs/Plant_Breed/test')
import cv2
for i in os.listdir():
  try:
    dummy = cv2.imread(i)
    dummy = cv2.resize(dummy,(128,128))
    x_test.append(dummy)
  except Exception as e:
    print(e)
    

In [ ]:
x_test_orig = x_test

In [ ]:
os.listdir()

In [ ]:
y_test = os.listdir()

In [ ]:
x_train = []
y_train = []
#y_test = []

In [ ]:
import os
os.chdir('/content/drive/My Drive/ComputerVision_with_CNNs/Plant_Breed')

In [ ]:
ls

In [ ]:
os.chdir('/content/drive/My Drive/ComputerVision_with_CNNs/Plant_Breed/train')

In [ ]:
os.listdir()

In [ ]:
l = os.listdir()

In [ ]:
le = preprocessing.LabelEncoder()
le.fit(l)

list(le.classes_)

le.transform(l) 

In [ ]:
list(le.inverse_transform([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]))

In [ ]:
x_train = []
y_train = []
import cv2
for i in os.listdir():
    print(i)
    if (os.path.isdir(i)):
            for j in os.listdir(i):
                try:
                    dummy = cv2.imread('/content/drive/My Drive/ComputerVision_with_CNNs/Plant_Breed/train/' + i + "/" + j)
                    dummy = cv2.resize(dummy,(128,128))
                    x_train.append(dummy)
                    y_train.append(i)
                except Exception as e:
                    print(e)


In [ ]:
x_train[0].shape

In [ ]:
import pandas as pd
dum = pd.get_dummies(y_train)

In [ ]:
encoded_labels = dum
y_train = dum

In [ ]:
import numpy as np
y_train = np.array(y_train)

In [ ]:
x_train = np.array(x_train)

In [ ]:
x_train[0].shape

In [ ]:
x_test = np.array(x_test)

In [ ]:
x_test[0].shape

#Divide the data set into Train and validation data sets

In [ ]:
from sklearn.model_selection import train_test_split
x_train2, x_val, y_train2, y_val = train_test_split(x_train, y_train, test_size=0.2, random_state=7)
print (len(x_train2))
print (len(x_val))

In [ ]:
x_train2[0].shape

In [ ]:
x_train2 = x_train2.reshape(x_train2.shape[0],128,128,3)

In [ ]:
x_val = x_val.reshape(x_val.shape[0],128,128,3)

In [ ]:
print (x_train2.shape)
print (x_val.shape)

In [ ]:
print(y_train2.shape)
print(y_val.shape)

#Initialize & build the model 

In [ ]:
import keras
from keras.models import Sequential
from keras.layers import Convolution2D, Dropout, Dense

In [ ]:
from keras.layers import BatchNormalization
from keras.layers import MaxPooling2D
from keras.layers import Flatten
from keras.optimizers import adam
from keras.optimizers import adamax
from keras.utils import np_utils

In [ ]:
model = Sequential()
model.add(BatchNormalization(input_shape = (128,128,3)))
model.add(Convolution2D(32, (3,3), activation ='relu', input_shape = (128, 128, 3)))
model.add(MaxPooling2D(pool_size=2))
#model.add(Dropout(0.2))

model.add(Convolution2D(filters=64, kernel_size=4, padding='same', activation='relu'))
model.add(MaxPooling2D(pool_size=2))
#model.add(Dropout(0.2))

model.add(Convolution2D(filters=128, kernel_size=3, padding='same', activation='relu'))
model.add(MaxPooling2D(pool_size=2))
#model.add(Dropout(0.2))

model.add(Convolution2D(filters=128, kernel_size=2, padding='same', activation='relu'))
model.add(MaxPooling2D(pool_size=2))
#model.add(Dropout(0.2))

model.add(Flatten()) 

# fully connected layer
model.add(Dense(units=128,activation = 'relu'))
model.add(Dense(units = 64, activation = 'relu'))
#model.add(Dropout(0.25))
model.add(Dense(units = 32, activation = 'relu'))
#model.add(Dropout(0.25))
model.add(Dense(units = 12, activation = 'softmax')) 


#Optimize the model

In [ ]:
optimizer = adam(lr=0.001)
model.compile(optimizer='adam', loss = 'categorical_crossentropy',metrics = ['accuracy'])

In [ ]:
model.summary()

In [ ]:
# Store Training Results
early_stopping = keras.callbacks.EarlyStopping(monitor='val_acc', patience=5, verbose=1, mode='auto')
callback_list = [early_stopping]

# Predict the accuracy for both train and validation data

In [ ]:
#batch_size = 64
batch_size = 32
nb_epochs = 10
history = model.fit(x_train,y_train,
                    epochs=50, 
                    validation_data=(x_val,y_val),
                    verbose = 1,
                    initial_epoch=0,
                    callbacks=callback_list)

In [ ]:
### Now, to the above model, lets add Data Augmentation 

### Import the ImageDataGenrator from keras and fit the training images

from keras.preprocessing.image import ImageDataGenerator

In [ ]:
x_train_255 = x_train / 255

In [ ]:
datagen = ImageDataGenerator(
    featurewise_center=False,
    featurewise_std_normalization=False,
    rotation_range=10, # random;y rotate images in range (degrees, 0 to 180)
    width_shift_range=0.1, # randomly shift images horizontally (fraction of total width)
    height_shift_range=0.1, # randomly shift images horizontally (fraction of total height)
    horizontal_flip=True)

# compute quantities required for featurewise normalization
# (std, mean, and principal components if ZCA whitening is applied)
datagen.fit(x_train_255)

# fits the model on batches with real-time data augmentation:
#model.fit_generator(datagen.flow(x_train, y_train, batch_size=32),
#                    steps_per_epoch=len(x_train) / 32, epochs=epochs)


In [ ]:
#### Showing 5 versions of the first image in training dataset using image datagenerator.flow()

from matplotlib import pyplot as plt
gen = datagen.flow(x_train_255[0:1], batch_size=1)
for i in range(1, 6):
    plt.subplot(1,5,i)
    plt.axis("off")
    plt.imshow(gen.next().squeeze(), cmap='gray')
    plt.plot()
plt.show()

In [ ]:
model = Sequential()
model.add(BatchNormalization(input_shape = (128,128,3)))

model.add(Convolution2D(32, (3,3), activation ='relu', input_shape = (128, 128, 3)))
model.add(MaxPooling2D(pool_size=2))
#model.add(Dropout(0.2))

model.add(Convolution2D(filters=64, kernel_size=4, padding='same', activation='relu'))

model.add(MaxPooling2D(pool_size=2))
#model.add(Dropout(0.2))

model.add(Convolution2D(filters=128, kernel_size=3, padding='same', activation='relu'))

model.add(MaxPooling2D(pool_size=2))
#model.add(Dropout(0.2))

model.add(Convolution2D(filters=128, kernel_size=2, padding='same', activation='relu'))

model.add(MaxPooling2D(pool_size=2))
#model.add(Dropout(0.2))

model.add(Flatten()) 

# fully connected layer
model.add(Dense(units=128,activation = 'relu'))

model.add(Dense(units = 64, activation = 'relu'))

#model.add(Dropout(0.25))
model.add(Dense(units = 32, activation = 'relu'))

#model.add(Dropout(0.25))
model.add(Dense(units = 12, activation = 'softmax')) 


In [ ]:
optimizer = adam(lr=0.001)
model.compile(optimizer='adam', loss = 'categorical_crossentropy',metrics = ['accuracy'])

In [ ]:
# Store Training Results
early_stopping = keras.callbacks.EarlyStopping(monitor='val_acc', patience=10, verbose=1, mode='auto')
callback_list = [early_stopping]

In [ ]:
#batch_size = 64
batch_size = 32
nb_epochs = 20
history = model.fit_generator(datagen.flow(x_train,y_train,batch_size=32),
                    samples_per_epoch=x_train.shape[0], nb_epoch=100, validation_data=(x_val, y_val),
                             callbacks=callback_list)

In [ ]:
#Train accuracy is 98.27%
#Validation accuracy is 98.42%

In [ ]:
# PREDICTIONS
y_pred = model.predict(x_test)
y_pred.shape

In [ ]:
y_pred

In [ ]:
y_class = np.argmax(y_pred, axis = 1) 
y_class.shape

In [ ]:
y_class

In [ ]:
list(le.inverse_transform(y_class))

In [ ]:
y_class_label = list(le.inverse_transform(y_class))

In [ ]:
os.chdir('/content/drive/My Drive/ComputerVision_with_CNNs/Plant_Breed')

In [ ]:
#import pdb; pdb.set_trace()
submission = pd.DataFrame({
        "file": y_test,
        "species": y_class_label
    })
submission.to_csv('Plant_Breed45.csv', index=False)
print('Exported')

In [ ]:
# After numerous runs the best Test score submitted to Kaggle was 96.221%

In [ ]:
os.chdir('/content/drive/My Drive/ComputerVision_with_CNNs/Plant_Breed')

In [ ]:
ls